# Import and Model Loading:

In [2]:
import sys
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from xray.interpretability import *

c:\Users\megha\Desktop\xray-transformers\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from xray.evaluation import (
    predict_target_logit,
    top_k_deletion_curve,
    random_deletion_curve,
    evaluate_logit_faithfulness
)

print("evaluation.py imported successfully")

evaluation.py imported successfully


In [5]:
#IMPORTS:

import torch
import matplotlib.pyplot as plt

from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizer
)

from xray.interpretability import DistilBertInterpreter

# LOAD THE HUGGINGFACE MODEL:

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
).to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    "JDsouza1/distilbert-imdb-sentiment"
)

print("Model loaded!")

Device: cpu


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 697.06it/s]


Model loaded!


In [6]:
interpreter = DistilBertInterpreter(
    model=model,
    tokenizer=tokenizer,
    device=device
)

# Testing with one sentence:

In [7]:
test_text = "This was not a bad movie at all"

(
    tokens,
    token_attributions,
    delta,
    input_output,
    baseline_output,
    total_attribution,
    completeness_error
) = interpreter.attribute(test_text)

In [8]:
with torch.no_grad():
    encoding = tokenizer(
        test_text,
        return_tensors="pt",
        truncation=True,
        padding=False
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    target_label = torch.argmax(
        outputs.logits,
        dim=1
    ).item()

print("Target label:", target_label)
print("Target logit:", input_output.item())

Target label: 1
Target logit: 0.7795644998550415


Aggregate token attributions:

In [9]:
words, word_scores = interpreter.aggregate_tokens(
    tokens,
    token_attributions
)

print("Word-level attributions:")

for word, score in zip(words, word_scores):
    print(f"{word:15} {float(score):+.4f}")

Word-level attributions:
this            +0.2466
was             +0.2090
not             +0.5315
a               +0.5419
bad             -0.4694
movie           -0.5103
at              +0.1927
all             -0.0304


# Attribution-guided deletion curve:

In [ ]:
from xray.evaluation import top_k_deletion_curve

top_k_curve = top_k_deletion_curve(
    model=model,
    tokenizer=tokenizer,
    text=test_text,
    tokens=tokens,
    scores=word_scores,
    target_label=target_label,
    device=device
)